In [70]:
import pandas as pd


loans_raw = pd.read_csv("data/library_loans_clean.csv")
members_raw = pd.read_csv("data/library_members.csv")

loans = loans_raw.copy()
members = members_raw.copy()

print(loans.shape)
print(members.shape)

(600, 8)
(148, 3)


In [71]:
loans.info()
members.info()

<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      600 non-null    str    
 1   member_id    600 non-null    str    
 2   title        600 non-null    str    
 3   genre        600 non-null    str    
 4   loan_date    597 non-null    str    
 5   return_date  564 non-null    str    
 6   late_fee     600 non-null    float64
 7   returned     600 non-null    bool   
dtypes: bool(1), float64(1), str(6)
memory usage: 33.5 KB
<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   member_id   148 non-null    str    
 1   branch      136 non-null    str    
 2   birth_year  130 non-null    float64
dtypes: float64(1), str(2)
memory usage: 3.6 KB


## Part 2: Go deeper

### 1.

- **There are 31 loans where the member_id is not in the member-table which is about 5,2% of all loans**
- **They stand for 347,5:- in late fees which is 4,6% of the total, so nothing out of the ordinary**
- **Their borrowing patterns regarding genre is nothing significant:**
- Children    10
- Sci-fi       5
- Crime        4
- Fantasy      4
- Romance      4
- History      4
- **I see no obvoius reason for why they (6 lenders) should be missing from the member table**

In [72]:
test_merge = loans.merge(members, on='member_id', how='left', validate="many_to_many", indicator=True)

test_merge["_merge"].value_counts()

_merge
both          585
left_only      31
right_only      0
Name: count, dtype: int64

In [73]:
print(31/600)

0.051666666666666666


In [74]:
mask = ~loans['member_id'].isin(members['member_id'])

print(loans.loc[mask, 'late_fee'].sum())
print(loans.loc[mask, 'late_fee'].sum() / loans["late_fee"].sum())

347.5
0.04641068447412354


In [75]:
loans["genre"].value_counts()

genre
Crime       155
Children    130
Sci-fi       89
Fantasy      87
Romance      82
History      57
Name: count, dtype: int64

In [76]:
loans.loc[mask, "genre"].value_counts()

genre
Children    10
Sci-fi       5
Crime        4
Fantasy      4
Romance      4
History      4
Name: count, dtype: int64

In [77]:
loans.loc[mask, "member_id"].unique()

<StringArray>
['M0097', 'M0068', 'M0056', 'M0070', 'M0016', 'M0004']
Length: 6, dtype: str

In [111]:
loans.loc[mask, ["loan_id", 
                 "member_id", 
                 "title", 
                 "genre", 
                 "loan_date", 
                 "return_date", 
                 "late_fee", 
                 "returned"]
                 ].sort_values("member_id", ascending=True).head(31)

,loan_id,member_id,title,genre,loan_date,return_date,late_fee,returned
517,L-30518,M0004,Harbour Lights,Romance,2026-06-03,2026-06-19,0.0,True
514,L-30515,M0004,The Memory Engine,Sci-fi,2026-06-04,2026-07-12,42.5,True
500,L-30501,M0004,A People Remembered,History,2026-05-31,2026-06-28,17.5,True
418,L-30419,M0004,Mimmi and the Moon,Children,2026-05-06,2026-05-18,0.0,True
559,L-30560,M0016,Salt and Iron,History,2026-06-16,2026-07-06,0.0,True
495,L-30496,M0016,Dead Reckoning Lane,Crime,2026-05-29,2026-07-01,30.0,True
124,L-30125,M0016,Between the Lines,Romance,2026-02-10,2026-02-19,0.0,True
376,L-30377,M0016,The Hanseatic Road,History,2026-04-25,2026-05-18,5.0,True
171,L-30172,M0016,The Summer House,Romance,2026-02-24,2026-03-03,0.0,True
576,L-30577,M0016,The Dragon Who Was Afraid,Children,2026-06-21,2026-07-02,0.0,True


### 2.

- **There is only one member that has not borrowed a book**
- **As for preference i think that the merge gives a fuller picture of the data, but if I was to use it in a program
the second version might provide a clearer way to follow the code, and doesnt need to create a new column just for calculation.**

In [90]:
test_merge = loans.merge(members, on='member_id', how='outer', validate="many_to_many", indicator=True)

test_merge["_merge"].value_counts()

_merge
both          585
left_only      31
right_only      1
Name: count, dtype: int64

In [85]:
members_in_loans = members['member_id'].isin(loans['member_id']).sum()
members_total = members['member_id'].count()

print(members_total - members_in_loans)

1


### 3.